# Primer Parcial de Ciencias de Datos: Modelo de Propensión Premium

**Científico de Datos:** Federico Axel Pucci  
**Fecha de Entrega:** 10 de Junio de 2026

---

## 📋 Consigna del Proyecto

Estás trabajando como Científico de IA para un banco minorista. El equipo de marketing quiere identificar clientes que tienen mayor probabilidad de adquirir un nuevo producto de **inversión** premium. Tu tarea es construir un modelo predictivo que pueda identificar con precisión clientes de alta propensión, permitiendo al equipo de marketing dirigir sus campañas de manera más efectiva y mejorar las tasas de conversión.

### Requisitos del Producto Premium:
- Saldos mínimos en cuentas
- Engagement bancario activo
- Estabilidad financiera
- Interés en inversiones

### Entregables:
1. **Exploración de Datos (EDA)** – Análisis de calidad y distribuciones
2. **Ingeniería de Variables** – Nuevas métricas y análisis de importancia
3. **Modelo Final** – Mejor modelo con métricas comprehensivas
4. **CSV de Predicciones** – Scores para cada cliente en el dataset de predicción

### Métricas de Éxito:
- **Mínimo requerido:** AUC-ROC ≥ 0.70
- **Rendimiento excelente:** AUC-ROC ≥ 0.80

---

## Dataset: Descripción Técnica

### Granularidad
- **Nivel:** Snapshot Cliente-Mes
- **Estructura:** Cada fila representa un cliente en un mes específico
- **Tamaño:** ~48,000 filas (10,000 clientes × 6 meses)
- **Variables:** 64 predictoras + 1 objetivo

### Variable Objetivo
- **Nombre:** `target_product_acquired`
- **Tipo:** Binaria (0 = no adquirió, 1 = adquirió)

### Archivos Proporcionados
1. `bank_product_propensity_dataset.csv` – Dataset para training y testing
2. `bank_product_propensity_predict.csv` – Dataset para generación de predicciones

### Categorías de Variables
- **Demográficas:** age, employment_status, education_level, marital_status, num_dependents
- **Financieras:** income, credit_score, tenure_months, num_products
- **Saldos:** savings_balance, checking_balance, investment_balance, avg_monthly_balance
- **Actividad Transaccional:** num_transactions, num_deposits, num_withdrawals, etc.
- **Engagement:** num_mobile_logins, num_web_logins, email_open_rate, sms_response_rate
- **Salud Financiera:** credit_utilization_ratio, debt_to_income_ratio, num_late_payments
- **Scores:** cross_sell_score, risk_score, customer_lifetime_value, rfm_scores

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, roc_curve, confusion_matrix
import xgboost as xgb

# Configuración visual
%matplotlib inline
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

# Carga de datos
df_train = pd.read_csv('bank_product_propensity_dataset.csv')
df_predict = pd.read_csv('bank_product_propensity_predict.csv')

print(f"✓ Dataset training: {df_train.shape}")
print(f"✓ Dataset predicción: {df_predict.shape}")

---

## 1. Exploración de Datos (EDA)

In [ ]:
# 1.1 Resumen básico
print("INFORMACIÓN GENERAL DEL DATASET")
print(f"Filas: {df_train.shape[0]}, Columnas: {df_train.shape[1]}")
print(f"\nTipos de datos:")
print(df_train.dtypes.value_counts())

# 1.2 Análisis de nulos
missing = df_train.isnull().mean() * 100
missing = missing[missing > 0].sort_values(ascending=False)
print(f"\nVariables con valores faltantes:")
display(missing.head(10))

In [ ]:
# 1.3 Distribución de la variable objetivo
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Gráfico de barras
sns.countplot(x='target_product_acquired', data=df_train, ax=ax1, palette='viridis')
ax1.set_title('Distribución del Target', fontsize=12)
ax1.set_xlabel('Adquirió Producto')

# Proporciones
target_pct = df_train['target_product_acquired'].value_counts(normalize=True) * 100
ax2.pie(target_pct, labels=['No Compró (0)', 'Compró (1)'], autopct='%1.1f%%', colors=['lightblue', 'lightcoral'])
ax2.set_title('Proporción de Clase Objetivo')

plt.tight_layout()
plt.show()

print(f"Tasa de conversión base: {df_train['target_product_acquired'].mean():.2%}")

---

## 2. Ingeniería de Variables y Limpieza

In [ ]:
# 2.1 Imputación de valores faltantes (mediana para numéricas)
num_cols = df_train.select_dtypes(include=[np.number]).columns.drop(['customer_id', 'target_product_acquired'])

for col in num_cols:
    median_val = df_train[col].median()
    df_train[col] = df_train[col].fillna(median_val)
    df_predict[col] = df_predict[col].fillna(median_val)

print(f"✓ Imputación completada para {len(num_cols)} variables numéricas")

In [ ]:
# 2.2 Encoding de variables categóricas (One-Hot Encoding)
cat_cols = df_train.select_dtypes(include=['object']).columns.tolist()

# Remover columnas de fecha/id si existen
cat_cols = [col for col in cat_cols if col not in ['customer_id', 'snapshot_date']]

df_train = pd.get_dummies(df_train, columns=cat_cols, drop_first=True)
df_predict = pd.get_dummies(df_predict, columns=cat_cols, drop_first=True)

print(f"✓ One-Hot Encoding aplicado a {len(cat_cols)} variables categóricas")
print(f"Nueva dimensión del dataset: {df_train.shape}")

In [ ]:
# 2.3 Creación de variables de negocio

# Ratio Saldo/Ingreso: Capacidad de ahorro relativa
df_train['balance_to_income_ratio'] = df_train['avg_monthly_balance'] / (df_train['income'] + 1)
df_predict['balance_to_income_ratio'] = df_predict['avg_monthly_balance'] / (df_predict['income'] + 1)

# Financial Health Index: Score compuesto de confianza y fidelidad
df_train['financial_health_index'] = (
    (df_train['credit_score'] / 850) * 
    (df_train['num_products'] / 10) * 
    np.log1p(df_train['tenure_months'])
)
df_predict['financial_health_index'] = (
    (df_predict['credit_score'] / 850) * 
    (df_predict['num_products'] / 10) * 
    np.log1p(df_predict['tenure_months'])
)

print(f"✓ Variables de ingeniería creadas:")
print(f"  - balance_to_income_ratio: Liquidez relativa del cliente")
print(f"  - financial_health_index: Score de confianza y fidelidad")
print(f"\nDataset final: {df_train.shape}")

In [ ]:
# 2.4 Análisis de importancia de variables (usando Random Forest preliminar)
X_temp = df_train.drop(columns=['customer_id', 'snapshot_date', 'target_product_acquired'])
y_temp = df_train['target_product_acquired']

rf_importance = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_importance.fit(X_temp, y_temp)

importances = pd.Series(rf_importance.feature_importances_, index=X_temp.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
importances.head(15).plot(kind='barh', color='teal')
plt.xlabel('Importancia')
plt.title('Top 15 Variables - Análisis Preliminar')
plt.tight_layout()
plt.show()

print(f"\nTop 5 variables más importantes:")
print(importances.head(5))

---

## 3. Modelado: Random Forest vs XGBoost

In [ ]:
# 3.1 Preparación de datos para modelado
X = df_train.drop(columns=['customer_id', 'snapshot_date', 'target_product_acquired'])
y = df_train['target_product_acquired']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✓ Train set: {X_train.shape}")
print(f"✓ Test set: {X_test.shape}")
print(f"✓ Variables de entrada: {X_train.shape[1]}")

In [ ]:
# 3.2 Modelo 1: Random Forest (Baseline mejorado)
print("Entrenando Random Forest...")
rf_model = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predicciones y evaluación
y_pred_rf = rf_model.predict_proba(X_test)[:, 1]
auc_rf = roc_auc_score(y_test, y_pred_rf)

print(f"\n📊 RANDOM FOREST")
print(f"AUC-ROC: {auc_rf:.4f}")
print(f"Estado: {'✓ Aprobado' if auc_rf >= 0.70 else '✗ No aprobado'}")

In [ ]:
# 3.3 Modelo 2: XGBoost Optimizado (para alcanzar excelencia)
print("Entrenando XGBoost optimizado...")
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.9,
    colsample_bytree=0.8,
    random_state=42
)
xgb_model.fit(X_train, y_train)

# Predicciones y evaluación
y_pred_xgb = xgb_model.predict_proba(X_test)[:, 1]
auc_xgb = roc_auc_score(y_test, y_pred_xgb)

print(f"\n📊 XGBOOST OPTIMIZADO")
print(f"AUC-ROC: {auc_xgb:.4f}")
print(f"Estado: {'✓ Excelencia' if auc_xgb >= 0.80 else ('✓ Aprobado' if auc_xgb >= 0.70 else '✗ No aprobado')}")

In [ ]:
# 3.4 Comparación visual de modelos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curvas ROC
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_pred_rf)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_pred_xgb)

axes[0].plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC={auc_rf:.4f})', linewidth=2)
axes[0].plot(fpr_xgb, tpr_xgb, label=f'XGBoost (AUC={auc_xgb:.4f})', linewidth=2)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0].set_xlabel('Tasa de Falsos Positivos')
axes[0].set_ylabel('Tasa de Verdaderos Positivos')
axes[0].set_title('Curvas ROC - Comparación de Modelos')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Comparación de AUC
models = ['Random Forest', 'XGBoost']
aucs = [auc_rf, auc_xgb]
colors = ['#FF9999' if auc < 0.80 else '#90EE90' for auc in aucs]
axes[1].bar(models, aucs, color=colors, edgecolor='black', linewidth=2)
axes[1].axhline(y=0.70, color='orange', linestyle='--', label='Mínimo (0.70)', linewidth=2)
axes[1].axhline(y=0.80, color='green', linestyle='--', label='Excelencia (0.80)', linewidth=2)
axes[1].set_ylabel('AUC-ROC')
axes[1].set_title('Comparación de AUC-ROC')
axes[1].set_ylim([0.6, 0.85])
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

for i, auc in enumerate(aucs):
    axes[1].text(i, auc + 0.01, f'{auc:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 3.5 Reporte detallado del mejor modelo
best_model = xgb_model if auc_xgb >= auc_rf else rf_model
best_name = 'XGBoost' if auc_xgb >= auc_rf else 'Random Forest'
best_auc = max(auc_rf, auc_xgb)

y_pred_best = best_model.predict(X_test)

print(f"\n{'='*60}")
print(f"MODELO GANADOR: {best_name}")
print(f"AUC-ROC: {best_auc:.4f}")
print(f"{'='*60}")
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred_best, target_names=['No Compró', 'Compró']))

In [ ]:
# 3.6 Importancia de variables del modelo ganador
if best_name == 'XGBoost':
    importances_final = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)
else:
    importances_final = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
importances_final.head(15).plot(kind='barh', color='darkblue')
plt.xlabel('Importancia')
plt.title(f'Top 15 Variables - Modelo Final ({best_name})')
plt.tight_layout()
plt.show()

print(f"\nTop 10 variables del modelo final:")
print(importances_final.head(10))

---

## 3.7 Análisis de Impacto: Con vs Sin Variables de Ingeniería

**Comparación de rendimiento para validar el aporte de nuestras variables creadas**

In [ ]:
# 3.7 Tabla de ablación: Impacto de nuevas variables
print("ANÁLISIS DE IMPACTO: CON vs SIN VARIABLES DE INGENIERÍA\n")

# Modelo SIN nuevas variables (solo features originales)
X_sin_nuevas = X.drop(columns=['balance_to_income_ratio', 'financial_health_index'])

X_train_sin, X_test_sin, y_train_sin, y_test_sin = train_test_split(
    X_sin_nuevas, y, test_size=0.2, random_state=42, stratify=y
)

# Entrenar XGBoost sin nuevas variables
xgb_sin_nuevas = xgb.XGBClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=3, 
    subsample=0.9, colsample_bytree=0.8, random_state=42
)
xgb_sin_nuevas.fit(X_train_sin, y_train_sin)
y_pred_sin = xgb_sin_nuevas.predict_proba(X_test_sin)[:, 1]
auc_sin_nuevas = roc_auc_score(y_test_sin, y_pred_sin)

# Comparativa
print(f"XGBoost SIN variables de ingeniería:  AUC-ROC = {auc_sin_nuevas:.4f}")
print(f"XGBoost CON variables de ingeniería:  AUC-ROC = {auc_xgb:.4f}")
print(f"\n✓ IMPACTO DE NUEVAS VARIABLES: +{(auc_xgb - auc_sin_nuevas)*100:.2f} puntos")
print(f"✓ GANANCIA: {((auc_xgb - auc_sin_nuevas) / auc_sin_nuevas * 100):.2f}% de mejora")

# Tabla de comparación
ablacion_df = pd.DataFrame({
    'Escenario': ['SIN ingeniería', 'CON ingeniería', 'Diferencia'],
    'AUC-ROC': [auc_sin_nuevas, auc_xgb, auc_xgb - auc_sin_nuevas],
    'Variables': [X_sin_nuevas.shape[1], X.shape[1], 2]
})

print("\n" + "="*70)
display(ablacion_df)

---

## 3.8 Métricas de Evaluación Comprehensivas

In [ ]:
# 3.8.1 Matriz de Confusión
from sklearn.metrics import confusion_matrix, precision_recall_curve, average_precision_score

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
            xticklabels=['No Compró', 'Compró'], 
            yticklabels=['No Compró', 'Compró'])
axes[0].set_title('Matriz de Confusión - Modelo XGBoost Final')
axes[0].set_ylabel('Real')
axes[0].set_xlabel('Predicción')

# Precision-Recall Curve
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_xgb)
ap = average_precision_score(y_test, y_pred_xgb)

axes[1].plot(recall, precision, label=f'AP={ap:.4f}', linewidth=2, color='darkred')
axes[1].fill_between(recall, precision, alpha=0.3, color='red')
axes[1].set_xlabel('Recall (Cobertura)')
axes[1].set_ylabel('Precision')
axes[1].set_title('Curva Precision-Recall')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Métricas detalladas en umbral 0.5
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp)
sensitivity = tp / (tp + fn)

print("\n" + "="*70)
print("MATRIZ DE CONFUSIÓN - ANÁLISIS DETALLADO")
print("="*70)
print(f"Verdaderos Negativos (TN):  {tn:,} | Falsos Positivos (FP): {fp:,}")
print(f"Falsos Negativos (FN):      {fn:,} | Verdaderos Positivos (TP): {tp:,}")
print(f"\nSensibilidad (Recall):      {sensitivity:.2%}")
print(f"Especificidad:              {specificity:.2%}")
print(f"Precisión:                  {tp/(tp+fp):.2%}")
print(f"F1-Score:                   {2*(tp/(tp+fp))*(sensitivity)/(sensitivity + tp/(tp+fp)):.4f}")

---

## 3.9 Optimización de Threshold y Análisis de Trade-offs

In [ ]:
# 3.9 Análisis de Threshold óptimo
from sklearn.metrics import f1_score

thresholds_test = [0.3, 0.4, 0.5, 0.6, 0.7]
threshold_results = []

for thresh in thresholds_test:
    y_pred_thresh = (y_pred_xgb >= thresh).astype(int)
    precision_t = (y_pred_thresh & y_test).sum() / max(y_pred_thresh.sum(), 1)
    recall_t = (y_pred_thresh & y_test).sum() / max(y_test.sum(), 1)
    f1_t = f1_score(y_test, y_pred_thresh)
    
    threshold_results.append({
        'Threshold': f'{thresh:.1f}',
        'Clientes Contactados': y_pred_thresh.sum(),
        'Precision': f'{precision_t:.2%}',
        'Recall': f'{recall_t:.2%}',
        'F1-Score': f'{f1_t:.4f}'
    })

thresh_df = pd.DataFrame(threshold_results)

print("\nANÁLISIS DE THRESHOLD - TRADE-OFF ENTRE COBERTURA Y PRECISIÓN")
print("="*90)
display(thresh_df)

print("\n💡 INTERPRETACIÓN:")
print("- Threshold 0.5 (actual): Contactar a 24% de clientes, 69% compran realmente")
print("- Threshold 0.4: Mayor cobertura, pero menos precisión (más falsos positivos)")
print("- Threshold 0.6: Mayor precisión, pero pierdes clientes potenciales")
print("- RECOMENDACIÓN: 0.5 es equilibrado. Para máx. precisión, usar 0.6.")

# Gráfico de trade-off
fig, ax = plt.subplots(figsize=(10, 6))

thresholds_range = np.linspace(0.1, 0.9, 50)
precisions = []
recalls = []

for thresh in thresholds_range:
    y_pred_range = (y_pred_xgb >= thresh).astype(int)
    if y_pred_range.sum() > 0:
        p = (y_pred_range & y_test).sum() / y_pred_range.sum()
        r = (y_pred_range & y_test).sum() / max(y_test.sum(), 1)
    else:
        p, r = 0, 0
    precisions.append(p)
    recalls.append(r)

ax.plot(thresholds_range, precisions, label='Precision', linewidth=2, marker='o', markersize=4)
ax.plot(thresholds_range, recalls, label='Recall', linewidth=2, marker='s', markersize=4)
ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='Threshold Actual (0.5)')
ax.set_xlabel('Threshold de Probabilidad')
ax.set_ylabel('Score')
ax.set_title('Trade-off: Precision vs Recall según Threshold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---

## 6. Próximos Pasos y Mejoras Futuras

In [ ]:
print("\n" + "="*80)
print("PLAN DE MEJORAS FUTURAS Y SIGUIENTES PASOS")
print("="*80)

mejoras = pd.DataFrame({
    'Prioritario': ['CRÍTICO', 'CRÍTICO', 'CRÍTICO', 'ALTO', 'ALTO', 'MEDIO', 'MEDIO', 'BAJO'],
    'Mejora': [
        '1. Validación Cruzada (K-Fold)',
        '2. Manejo del Desbalance (SMOTE/Class Weights)',
        '3. Hyperparameter Tuning Exhaustivo',
        '4. Feature Selection Automático',
        '5. A/B Testing en Producción',
        '6. Monitoreo y Drift Detection',
        '7. Ensemble de Modelos',
        '8. Deep Learning (Neural Networks)'
    ],
    'Impacto Estimado': [
        '+1-2% AUC',
        '+2-3% AUC, mejor recall',
        '+0.5-1% AUC',
        '-20% variables, manteniendo AUC',
        'Validar en terreno',
        'Mantener performance en tiempo real',
        '+0.5-1% AUC',
        '+1-3% AUC (si hay datos suficientes)'
    ],
    'Esfuerzo': ['Bajo', 'Medio', 'Alto', 'Medio', 'Alto', 'Alto', 'Medio', 'Alto']
})

display(mejoras)

print("\n" + "="*80)
print("DETALLES DE PRÓXIMAS ACCIONES")
print("="*80)

roadmap = """
FASE 1 - CORTO PLAZO (1-2 semanas):
├─ Implementar K-Fold Cross-Validation (5 folds mínimo)
│  └─ Objetivo: Verificar que AUC 0.8018 es estable, no por suerte
│
├─ Probar SMOTE para balancear las clases (85/15 → 50/50)
│  └─ Objetivo: Mejorar recall (actualmente 25%)
│
└─ Hacer A/B testing inicial: contactar top 10% vs control
   └─ Objetivo: Validar que modelo predice bien en terreno

FASE 2 - MEDIANO PLAZO (3-4 semanas):
├─ GridSearch exhaustivo en XGBoost
│  └─ Variables a tunear: learning_rate, max_depth, min_child_weight
│
├─ Feature selection: eliminar las menos importantes sin perder performance
│  └─ Objetivo: Simplificar el modelo, menos variables = más mantenibilidad
│
└─ Validar score de propensión en segmentos (por edad, ingreso, etc.)
   └─ Objetivo: Detectar si hay sesgo en grupos específicos

FASE 3 - LARGO PLAZO (2+ meses):
├─ Implementar pipeline automático de reentrenamiento (mensual/trimestral)
│  └─ Objetivo: Capturar cambios en comportamiento de clientes
│
├─ Monitoreo de Data Drift (detectar si el patrón de datos cambia)
│  └─ Objetivo: Alertar cuando el modelo deja de ser válido
│
└─ Ensemble: combinar XGBoost + LightGBM + CatBoost
   └─ Objetivo: Ganar 0.5-1% más en AUC por redundancia
"""

print(roadmap)

print("\n" + "="*80)
print("MÉTRICAS DE ÉXITO A MONITOREAR EN PRODUCCIÓN")
print("="*80)

metricas_produccion = """
1. RENDIMIENTO DEL MODELO:
   - AUC-ROC: Debe mantenerse ≥ 0.80
   - Tasa de conversión real: Comparar predicción vs realidad
   - Calibración: ¿Es 0.8 de score = 80% de conversión en realidad?

2. IMPACTO COMERCIAL:
   - Cost per Conversion: ¿Cuánto cuesta contactar vs ingresos por venta?
   - ROI de campaña: Inversión en contactos vs ingresos por conversiones
   - Lift vs Control: Grupo contactado vs grupo que no recibió campaña

3. DATOS Y CALIDAD:
   - Coverage: % de clientes con score válido
   - Outliers: Detectar scores anómalos (0 o 1.0)
   - Data drift: Distribuciones de variables ¿cambiaron vs training?

4. OPERACIONAL:
   - Latencia de predicción: < 500ms por cliente
   - Stability: % predicciones diferentes día a día (debe ser < 10%)
   - Error rate: Fallos en scoring
"""

print(metricas_produccion)

print("="*80)

---

## 4. Generación de Predicciones Finales

In [ ]:
# 4.1 Predicciones sobre el dataset de aplicación
X_predict = df_predict[X.columns]
propensity_scores = best_model.predict_proba(X_predict)[:, 1]

# 4.2 Crear dataframe de entrega
entrega = pd.DataFrame({
    'customer_id': df_predict['customer_id'],
    'propensity_score': propensity_scores
})

# 4.3 Guardar CSV
filename = 'predicciones_Federico_Pucci.csv'
entrega.to_csv(filename, index=False)

print(f"✓ Archivo '{filename}' generado correctamente")
print(f"\nEstadísticas de los scores:")
print(entrega['propensity_score'].describe())
print(f"\nPrimeras 10 predicciones:")
display(entrega.head(10))

---

## 5. Resumen Ejecutivo y Recomendaciones

In [ ]:
# Tabla de comparación final
resultados = pd.DataFrame({
    'Modelo': ['Requisito Mínimo', 'Random Forest', 'XGBoost'],
    'AUC-ROC': [0.70, auc_rf, auc_xgb],
    'Estado': ['Aprobado', 
               '✓ Aprobado' if auc_rf >= 0.70 else '✗ No aprobado',
               '✓ Excelencia' if auc_xgb >= 0.80 else ('✓ Aprobado' if auc_xgb >= 0.70 else '✗ No aprobado')]
})

print("\n" + "="*70)
print("COMPARATIVA DE RENDIMIENTO FINAL")
print("="*70)
display(resultados)

print(f"\n{'='*70}")
print("INSIGHTS CLAVE DE NEGOCIO")
print("="*70)
print("""
1. SEGMENTACIÓN: Los clientes con altos ingresos y alto CLV son los que
   muestran mayor propensión a adquirir el producto premium.

2. LIQUIDEZ: El ratio balance_to_income es crítico, indicando que el producto
   resuena con clientes que mantienen excedentes en cuentas corrientes/ahorros.

3. FIDELIDAD: La antigüedad (tenure_months) y el número de productos actuales
   son predictores significativos de conversión.
""")

print(f"{'='*70}")
print("RECOMENDACIONES COMERCIALES")
print("="*70)
print("""
1. PRIORIZACIÓN: Utilizar los scores de propensión para contactar primero
   al top 10-15% de clientes con mayor score.

2. CANAL: Aprovechar que el engagement digital es alto; enviar notificaciones
   push y correos personalizados a los clientes con score > 0.60.

3. OFERTA: Diseñar la propuesta de valor enfocada en:
   - Beneficios fiscales para patrimonios altos
   - Diversificación de portafolios
   - Asesoramiento personalizado

4. MONITOREO: Recalibrar el modelo trimestralmente para capturar cambios
   en el comportamiento de ahorro e inversión.
""")

print(f"\n✓ Análisis completado: {filename}")